In [1]:
from typing import Callable, Tuple
from qiskit import QuantumCircuit
import qiskit
from qiskit import transpile
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Operator
import math
from math import pi
import numpy as np

from sqlalchemy.orm import joinedload
from sqlalchemy import select

from benchmarklib import BenchmarkDatabase
from benchmarklib.runners import BatchQueue
from benchmarklib.pipeline.synthesis import XAGSynthesizer, TruthTableSynthesizer

from experiments import ExperimentProblem, ExperimentTrial

TAG = "eqs_comparison"

In [2]:
service = QiskitRuntimeService()
backend = service.backend("ibm_rensselaer")
db = BenchmarkDatabase("experiments.db", ExperimentProblem, ExperimentTrial)

In [3]:
def generate_log_n_and(N):
    """
    Generates the verifier for the AND of two N-qubit numbers using a logarithmic depth of AND operations.
    Assumption: input[0:N] = A, input[N:2N] = B
    """
    #res = [not (a^b) for (a,b) in zip(A,B)]
    assert (N & (N-1)) == 0, "N must be a power of 2"
    output = []
    output.append("def verify(inpt: Tuple[bool]) -> bool:")
    for c in range(N):
        output.append("res_0_"+str(c)+" = not (inpt["+str(c)+"] ^ inpt["+str(N+c)+"])")
    step = 1
    while step <= N/2:
        i = 0
        while i < N:
            #res[i] = res[i] & res[i+step]
            output.append("res_"+str(step)+"_"+str(i)+" = "+"res_"+str(step//2)+"_"+str(i)+" & "+"res_"+str(step//2)+"_"+str(i+step)) 
            i = i + step*2
        step = step*2
    output.append("return res_"+str(int(N/2))+"_0")
    return '\n    '.join(output)    

def generate_seq_and(N):
    """
    Generates the verifier for the AND of two N-qubit numbers using a linear depth of AND operations.
    Assumption: input[0:N] = A, input[N:2N] = B
    """
    #nxor = [not (a^b) for (a,b) in zip(A,B)]
    output = []
    output.append("def verify(inpt: Tuple[bool]) -> bool:")
    for c in range(N):
        #nxor.append(A[c]^B[c])
        output.append("nxor_"+str(c)+" = not (inpt["+str(c)+"] ^ inpt["+str(N+c)+"])")
    #res = True
    output.append("res_0 = True")
    for i in range(1,N+1):
        #res = res & nxor[i-1]
        output.append("res_"+str(i)+" = res_"+str(i-1)+" & nxor_"+str(i-1))
    output.append("return res_"+str(N))
    return '\n    '.join(output)    

In [4]:
# build the circuits we are comparing
problems = {
    "log_and4": db.get_or_create(ExperimentProblem, name="log_and4", tag=TAG, n=2*4),
    "seq_and4": db.get_or_create(ExperimentProblem, name="seq_and4", tag=TAG, n=2*4),
    "log_and8": db.get_or_create(ExperimentProblem, name="log_and8", tag=TAG, n=2*8),
    "seq_and8": db.get_or_create(ExperimentProblem, name="seq_and8", tag=TAG, n=2*8),
}

problems["log_and4"].verifier_src = generate_log_n_and(4)
problems["seq_and4"].verifier_src = generate_seq_and(4)
problems["log_and8"].verifier_src = generate_log_n_and(8)
problems["seq_and8"].verifier_src = generate_seq_and(8)

candidates = list(problems.keys())

base_circuits = {
    name : XAGSynthesizer().synthesize(problems[name]) for name in candidates
}

0
4
1
5
2
6
3
7
0
4
1
5
2
6
3
7
0
8
1
9
2
10
3
11
4
12
5
13
6
14
7
15
0
8
1
9
2
10
3
11
4
12
5
13
6
14
7
15


In [55]:
print("Pre-transpile Metrics")
for candidate in candidates:
    circuit = base_circuits[candidate]
    print(f"{candidate}: Depth: {circuit.depth()}; ops: {circuit.count_ops()}")
    

Pre-transpile Metrics
log_and4: Depth: 6; ops: OrderedDict([('cx', 16), ('ccrx_o0', 4), ('ccx', 1)])
seq_and4: Depth: 8; ops: OrderedDict([('cx', 14), ('ccrx_o0', 2), ('ccrx_o2', 2), ('ccx_o2', 1)])
log_and8: Depth: 7; ops: OrderedDict([('cx', 32), ('ccrx_o0', 8), ('ccrx', 4), ('ccx', 1)])
seq_and8: Depth: 16; ops: OrderedDict([('cx', 30), ('ccrx_o2', 10), ('ccrx_o0', 2), ('ccx_o2', 1)])


In [15]:
def create_trials(problem: ExperimentProblem):
    trials = []
    N = problem.n // 2
    # generate a variety of input states to test with
    input_states = []
    for i in range(8):
        matching_val = np.random.randint(0, 2**N - 1)
        input_state = format(matching_val, '0' + str(N) + 'b') * 2
        input_states.append(input_state)
    for i in range(8):
        random_val = np.random.randint(0, 2**(2*N) - 1)
        input_state = format(random_val, '0' + str(2*N) + 'b')
        input_states.append(input_state)

    circuit_width = max(problem.n + 1, base_circuits[problem.name].width())
    
    for input_state in input_states:
        
        qc = QuantumCircuit(circuit_width, problem.n + 1)
        for qubit in range(problem.n):
            if input_state[qubit] == '1':
                qc.x(qubit)

        qc.compose(base_circuits[problem.name], inplace=True)
        qc.measure(range(problem.n + 1), range(problem.n + 1))

        qc_final = transpile(qc, backend=backend, optimization_level=3)

        # calculate expected output state
        result = "1" if input_state[0:N] == input_state[N:2*N] else "0"
        expected_output = result + input_state[::-1]
        
        trials.append(
            ExperimentTrial(
                problem=problem,
                circuit=qc_final,
                circuit_pretranspile=qc,
                extra_data={"input_state": input_state, "backend": backend.name, "expected_output": expected_output}
            )
        )
    return trials

In [ ]:
# sanity check one of the trials
trial = create_trials(problems["log_and4"])[0]
simulator = AerSimulator()
results = simulator.run(trial.circuit, shots=4096).result()
counts = results.get_counts()
print("Input State:", trial.extra_data["input_state"])
print("Expected Output:", trial.extra_data["expected_output"])
print("Counts:", counts)

Input State: 01110111
Expected Output: 111101110
Counts: {'111101110': 4096}


In [54]:
for candidate in candidates:
    problem = problems[candidate]
    trials = create_trials(problem)
    print(f"{candidate}: Depth: {trials[0].circuit.depth()}; ops: {trials[0].circuit.count_ops()}")


log_and4: Depth: 138; ops: OrderedDict([('rz', 206), ('sx', 113), ('ecr', 73), ('x', 23), ('measure', 9)])
seq_and4: Depth: 213; ops: OrderedDict([('rz', 190), ('sx', 107), ('ecr', 69), ('x', 22), ('measure', 9)])
log_and8: Depth: 292; ops: OrderedDict([('rz', 644), ('sx', 361), ('ecr', 223), ('x', 45), ('measure', 17)])
seq_and8: Depth: 726; ops: OrderedDict([('rz', 646), ('sx', 382), ('ecr', 227), ('x', 35), ('measure', 17)])


In [38]:
with BatchQueue(db, backend=backend, shots=4096) as q:
    for candidate in candidates:
        problem = problems[candidate]
        trials = create_trials(problem)
        for trial in trials:
            q.enqueue(trial, trial.circuit, run_simulation=False)

In [46]:
await db.update_all_pending_results(service=service)

### Visualize Results

In [47]:
def calculate_fidelity(trial):
    expected_output = trial.extra_data["expected_output"]
    total_shots = sum(trial.counts.values())
    correct_shots = trial.counts.get(expected_output, 0)
    fidelity = correct_shots / total_shots
    return fidelity

In [48]:
trials = db.query(
    select(ExperimentTrial).join(ExperimentTrial.problem).options(joinedload(ExperimentTrial.problem))
    .where(ExperimentProblem.tag == TAG, ExperimentTrial.counts != None)
)

In [49]:
fidelities = {candidate: [] for candidate in candidates}
for trial in trials:
    fidelity = calculate_fidelity(trial)
    fidelities[trial.problem.name].append(fidelity)

In [50]:
for candidate in candidates:
    avg_fidelity = sum(fidelities[candidate]) / len(fidelities[candidate])
    print(f"{candidate}: Average Fidelity = {avg_fidelity:.5f}")

log_and4: Average Fidelity = 0.02493
seq_and4: Average Fidelity = 0.03375
log_and8: Average Fidelity = 0.00019
seq_and8: Average Fidelity = 0.00010


In [51]:
fidelities_positive = {candidate: [] for candidate in candidates}
fidelities_negative = {candidate: [] for candidate in candidates}
for trial in trials:
    fidelity = calculate_fidelity(trial)
    if trial.extra_data["expected_output"][0] == '1':
        fidelities_positive[trial.problem.name].append(fidelity)
    else:
        fidelities_negative[trial.problem.name].append(fidelity)


In [52]:
for candidate in candidates:
    avg_pos_fidelity = sum(fidelities_positive[candidate]) / len(fidelities_positive[candidate])
    avg_neg_fidelity = sum(fidelities_negative[candidate]) / len(fidelities_negative[candidate])
    print(f"{candidate}: Fidelity (Average Positive) = {avg_pos_fidelity:.8f}, Fidelity (Average Negative) = {avg_neg_fidelity:.8f}")

log_and4: Fidelity (Average Positive) = 0.02298607, Fidelity (Average Negative) = 0.02731536
seq_and4: Fidelity (Average Positive) = 0.03128685, Fidelity (Average Negative) = 0.03678575
log_and8: Fidelity (Average Positive) = 0.00023905, Fidelity (Average Negative) = 0.00014750
seq_and8: Fidelity (Average Positive) = 0.00011190, Fidelity (Average Negative) = 0.00009155
